In [ ]:
!pip install -qU langgraph langchain-google-genai langchain-core chromadb pandas pydantic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [ ]:
import os
import pandas as pd
import chromadb
from google.colab import userdata

# Load Gemini API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

# 1. Create realistic synthetic bank statement CSV
transactions = [
    {"date": "2026-09-02", "merchant": "StreamFlix India", "amount": 649.0, "category": "Entertainment"},
    {"date": "2026-10-02", "merchant": "StreamFlix India", "amount": 649.0, "category": "Entertainment"},
    {"date": "2026-11-02", "merchant": "StreamFlix India", "amount": 899.0, "category": "Entertainment"}, # Price hike!
    {"date": "2026-09-10", "merchant": "CloudFit Gym App", "amount": 499.0, "category": "Health"},
    {"date": "2026-10-10", "merchant": "CloudFit Gym App", "amount": 499.0, "category": "Health"},
    {"date": "2026-11-10", "merchant": "CloudFit Gym App", "amount": 499.0, "category": "Health"}, # Recurring bill
    {"date": "2026-11-14", "merchant": "QuickBite Delivery", "amount": 380.0, "category": "Food"},
    {"date": "2026-11-14", "merchant": "QuickBite Delivery", "amount": 380.0, "category": "Food"}, # Duplicate charge!
    {"date": "2026-11-18", "merchant": "Metro Rail Card", "amount": 250.0, "category": "Transport"},
]
df = pd.DataFrame(transactions)
df.to_csv("bank_statement.csv", index=False)

# 2. Initialize ChromaDB with merchant cancellation/refund policies
chroma_client = chromadb.Client()
policy_collection = chroma_client.get_or_create_collection(name="merchant_policies")

policies = [
    "StreamFlix India: Price increases require 30-day advance notice. Customers can request a pro-rata refund or downgrade via billing@streamflix.in within 14 days of a hike.",
    "CloudFit Gym App: Monthly subscriptions renew automatically unless cancelled 48 hours prior via support@cloudfit.in.",
    "QuickBite Delivery: Duplicate same-day charges are eligible for instant automated reversal by emailing disputes@quickbite.in with transaction dates."
]
policy_collection.add(documents=policies, ids=["p1", "p2", "p3"])

print("✅ Setup complete: bank_statement.csv and ChromaDB policy store are ready!")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 37.1MiB/s]


✅ Setup complete: bank_statement.csv and ChromaDB policy store are ready!


In [ ]:
from datetime import datetime, timedelta
from langchain_core.tools import tool

@tool
def audit_bank_statement(csv_path: str) -> dict:
    """Analyzes a bank statement CSV using Pandas to detect recurring subscriptions, sudden price hikes, and duplicate same-day charges."""
    data = pd.read_csv(csv_path)
    findings = {"recurring_subscriptions": [], "price_hikes": [], "duplicate_charges": []}

    for merchant, group in data.groupby("merchant"):
        group = group.sort_values("date")
        if len(group) >= 2 and group["date"].nunique() >= 2:
            amounts = group["amount"].tolist()
            last_date = group["date"].iloc[-1]
            findings["recurring_subscriptions"].append({
                "merchant": merchant,
                "charges": amounts,
                "latest_billing_date": last_date,
                "count": len(group)
            })
            if amounts[-1] > amounts[0]:
                pct_increase = round(((amounts[-1] - amounts[0]) / amounts[0]) * 100, 1)
                findings["price_hikes"].append({
                    "merchant": merchant,
                    "old_amount": amounts[0],
                    "new_amount": amounts[-1],
                    "increase_pct": pct_increase
                })

    dupes = data[data.duplicated(subset=["date", "merchant", "amount"], keep=False)]
    if not dupes.empty:
        for (date, merchant, amount), _ in dupes.groupby(["date", "merchant", "amount"]):
            findings["duplicate_charges"].append({
                "date": date, "merchant": merchant, "amount": amount
            })

    return findings

@tool
def lookup_merchant_policy(merchant_name: str) -> str:
    """Searches ChromaDB for cancellation, refund, or dispute policies for a specific merchant."""
    results = policy_collection.query(query_texts=[merchant_name], n_results=1)
    if results["documents"] and results["documents"][0]:
        return results["documents"][0][0]
    return "No specific policy found. Use standard customer support dispute procedure."

@tool
def schedule_renewal_reminder(merchant_name: str, last_billing_date: str, amount: float) -> str:
    """Generates an .ics calendar reminder file 2 days before the next expected monthly renewal date."""
    last_date = datetime.strptime(last_billing_date, "%Y-%m-%d")
    next_billing = last_date + timedelta(days=30)
    reminder_date = next_billing - timedelta(days=2)

    ics_content = f"""BEGIN:VCALENDAR
VERSION:2.0
BEGIN:VEVENT
SUMMARY:Cancel/Review {merchant_name} Subscription (Rs.{amount})
DTSTART;VALUE=DATE:{reminder_date.strftime('%Y%m%d')}
DTEND;VALUE=DATE:{reminder_date.strftime('%Y%m%d')}
DESCRIPTION:Automated reminder from Bill Auditor Agent. Next charge of Rs.{amount} due on {next_billing.strftime('%Y-%m-%d')}.
END:VEVENT
END:VCALENDAR"""

    filename = f"reminder_{merchant_name.lower().replace(' ', '_')}.ics"
    with open(filename, "w") as f:
        f.write(ics_content)

    return f"Created calendar reminder file '{filename}' for {reminder_date.strftime('%Y-%m-%d')} (2 days before next Rs.{amount} renewal)."

@tool
def query_spending_stats(csv_path: str, category: str = None) -> dict:
    """Calculates total spending, category breakdowns, and merchant totals from the bank statement CSV."""
    data = pd.read_csv(csv_path)
    if category:
        filtered = data[data["category"].str.lower() == category.lower()]
        return {
            "category": category,
            "total_spent": float(filtered["amount"].sum()),
            "transaction_count": int(len(filtered)),
            "transactions": filtered.to_dict(orient="records")
        }
    return {
        "overall_total": float(data["amount"].sum()),
        "by_category": data.groupby("category")["amount"].sum().to_dict(),
        "by_merchant": data.groupby("merchant")["amount"].sum().to_dict()
    }

tools = [audit_bank_statement, lookup_merchant_policy, schedule_renewal_reminder, query_spending_stats]
print("✅ All 4 agent tools registered successfully!")

✅ All 4 agent tools registered successfully!


In [ ]:
from typing import Annotated, TypedDict, Literal
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

class DisputeEmail(BaseModel):
    recipient: str = Field(description="Merchant support/billing email address")
    subject: str = Field(description="Clear dispute or cancellation email subject")
    body: str = Field(description="Polite, factual email referencing exact dates and amounts")

class AuditorState(TypedDict):
    messages: Annotated[list, add_messages]
    draft_email: dict
    final_status: str

# Retry-safe free-tier Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", max_retries=5)
llm_with_tools = llm.bind_tools(tools)

def auditor_node(state: AuditorState):
    sys_prompt = SystemMessage(content=(
        "You are an autonomous Financial & Subscription Auditor. "
        "1. Call `audit_bank_statement` on 'bank_statement.csv'. "
        "2. For any price hikes or duplicate charges, call `lookup_merchant_policy` to get dispute rules. "
        "3. For recurring subscriptions without price hikes (like CloudFit Gym App), call `schedule_renewal_reminder` using its `latest_billing_date` from the audit results. "
        "Finally, summarize all findings, created calendar reminder files, and potential annual savings clearly."
    ))
    response = llm_with_tools.invoke([sys_prompt] + state["messages"])
    return {"messages": [response]}

def route_auditor(state: AuditorState) -> Literal["tools", "draft_action"]:
    last_msg = state["messages"][-1]
    if last_msg.tool_calls:
        return "tools"
    return "draft_action"

def draft_action_node(state: AuditorState):
    structured_llm = llm.with_structured_output(DisputeEmail)
    prompt = (
        "Based on the audit findings and merchant policies in our conversation, "
        "draft ONE highest-priority dispute or refund email (e.g., for the StreamFlix price hike or QuickBite duplicate charge)."
    )
    email_obj = structured_llm.invoke(state["messages"] + [HumanMessage(content=prompt)])
    return {"draft_email": email_obj.model_dump()}

def human_approval_node(state: AuditorState):
    decision = interrupt({
        "message": "Review drafted dispute email before dispatching:",
        "draft_email": state["draft_email"]
    })
    if decision.strip().lower() == "approve":
        return {"final_status": "APPROVED"}
    return {"final_status": "REJECTED"}

def dispatch_email_node(state: AuditorState):
    if state.get("final_status") != "APPROVED":
        return {"final_status": "Cancelled by user. No email sent."}
    email = state["draft_email"]
    return {"final_status": f"DISPATCHED email to {email['recipient']} | Subject: '{email['subject']}'"}

# Assemble LangGraph workflow
builder = StateGraph(AuditorState)
builder.add_node("auditor", auditor_node)
builder.add_node("tools", ToolNode(tools))
builder.add_node("draft_action", draft_action_node)
builder.add_node("human_gate", human_approval_node)
builder.add_node("dispatch", dispatch_email_node)

builder.add_edge(START, "auditor")
builder.add_conditional_edges("auditor", route_auditor)
builder.add_edge("tools", "auditor")
builder.add_edge("draft_action", "human_gate")
builder.add_edge("human_gate", "dispatch")
builder.add_edge("dispatch", END)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

# Run Step 1: Audit + Calendar Reminder + Pause at Human Gate
config = {"configurable": {"thread_id": "audit-session-01"}}
result = graph.invoke(
    {"messages": [HumanMessage(content="Audit my bank_statement.csv for hidden subscriptions, price hikes, or duplicate charges, and set calendar reminders for recurring bills.")]},
    config=config
)

print("=== AUDIT & CALENDAR REMINDER SUMMARY ===")
print(result["messages"][-1].content)
print("\n=== PAUSED AT HUMAN-IN-THE-LOOP GATE ===")
print(result["__interrupt__"][0].value)

=== AUDIT & CALENDAR REMINDER SUMMARY ===
[{'type': 'text', 'text': "Here's a summary of your bank statement audit:\n\n**Audit Findings:**\n\n*   **Duplicate Charge:** A duplicate charge of **Rs. 380** was identified for **QuickBite Delivery** on 2026-11-14.\n    *   **Dispute Policy:** QuickBite Delivery allows instant automated reversal for duplicate same-day charges. You can initiate this by emailing disputes@quickbite.in with the transaction dates.\n*   **Price Hike:** Your **StreamFlix India** subscription increased by 38.5%, from Rs. 649 to **Rs. 899** per month.\n    *   **Dispute Policy:** StreamFlix India requires 30-day advance notice for price increases. You can request a pro-rata refund or downgrade your plan by contacting billing@streamflix.in within 14 days of the price hike.\n*   **Recurring Subscription:** Your **CloudFit Gym App** subscription is a recurring charge of **Rs. 499** per month. No price hike was detected for this subscription.\n\n**Calendar Reminder Create

In [ ]:
user_choice = input("Type 'approve' to send email or 'reject' to cancel: ")
final_result = graph.invoke(Command(resume=user_choice), config=config)

print("\n=== FINAL EXECUTION STATUS ===")
print(final_result["final_status"])

Type 'approve' to send email or 'reject' to cancel: approve

=== FINAL EXECUTION STATUS ===
DISPATCHED email to billing@streamflix.in | Subject: 'Dispute Regarding Unauthorized Price Increase - Account [Your StreamFlix Account ID]'


In [ ]:
from langchain_core.messages import ToolMessage

qa_tools = [audit_bank_statement, query_spending_stats, lookup_merchant_policy]
qa_llm = llm.bind_tools(qa_tools)
tool_map = {t.name: t for t in qa_tools}

# Load full conversation memory from the completed LangGraph audit session
saved_state = graph.get_state(config).values
chat_history = list(saved_state.get("messages", []))

print("=== INTERACTIVE FINANCIAL Q&A MODE ===")
print("Ask any question about your statement, subscriptions, or savings (type 'exit' to stop).\n")

while True:
    user_q = input("You: ")
    if user_q.strip().lower() in ["exit", "quit", "stop"]:
        print("Q&A session ended.")
        break

    chat_history.append(HumanMessage(content=user_q))
    ai_msg = qa_llm.invoke(chat_history)
    chat_history.append(ai_msg)

    if ai_msg.tool_calls:
        for tc in ai_msg.tool_calls:
            tool_result = tool_map[tc["name"]].invoke(tc["args"])
            chat_history.append(ToolMessage(content=str(tool_result), tool_call_id=tc["id"]))
        final_reply = qa_llm.invoke(chat_history)
        chat_history.append(final_reply)
        print(f"\nAgent: {final_reply.content}\n")
    else:
        print(f"\nAgent: {ai_msg.content}\n")

=== INTERACTIVE FINANCIAL Q&A MODE ===
Ask any question about your statement, subscriptions, or savings (type 'exit' to stop).

You: which is the most expensive subscription i have?

Agent: [{'type': 'text', 'text': 'Based on your bank statement, the most expensive subscription you have is **StreamFlix India** at **Rs. 899** per month.', 'extras': {'signature': 'CvMFAWkUfRP39Hs6TQXm4C2aox5Kr/D+iPAoRTcp68NL5iLSYa58rY2tc3+ax948ezArm91mE4pYgXp/splbrboTwKC/t1wjM65S9GsbbWa+YeXLzaQWeho3cDRv4kD0IoIq2lo8fUBEsz0cl8o0Y87PZzytEo+cZJmG/oC6lf9vqESk4xxm7fJjHpF3CLK3yl0681CwqlzDQKFlbvbpfW89IXN7t3f0V/qFw4dSrlpi9lYJA3y9E+K1CIwQLtkC4LpL4ybhfgnDxjwXpo71Phxg6frHZ8bwBpdWrwsJVvFuJG0wpkrxcyamLqgYCfxOaf1aL6W0HLd53WGCsPhvC1nfQdeQH0G4wXtM3PBZKd48ioUIv2D1jB9IQPXpHGmvqsvv1PbyJcwWiA8DzcOstpnz/D36KTg3xsbnMB+XW3MVVqRQhi9okxzuWfZJbRvOlg8v4q9k8KBiOwo5FQnEmpGOSZMrw63iStrCpMAlMaJIqhg0DNLV7MEKMKC1AAfV1FG6d2C4UXAojTHHT9oZqGFWREFjvjWEQLFmw9mX1Rd98Q4gewIGau39Hzfy1eU1r/2cyp9FQiVFSeJ2Jo7B8D18B9Gd0YjWnheeTDOQlrLsR87j9fd+gSnJjWi